# Detect Occurences of Cytosine in DNA Regions

This is a program to count the occurences of cytosine in a given DNA sequence. It can be used to count the occurences of ctosinie both inside and outside of amyloidogenic regions.

In [ ]:
#mounting google drive, will link to account this is being run on

from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
!pip install Bio

#The code below defines the functions used to conduct downstream analysis.
count_cytosines calculates the percentage of each nucleotide and stores it in a dictionary with gene names. Two separate columns are created for each nucleotide, 'prion forming' and 'non-prion forming'. If the gene/protein is prion forming, the nucleotide is scored in the respective column and is absent in the non-prion forming list.

position_bias counts the percentage of cytosine residues in the first, second and third positions of nucleotides. These values are appended as additional columns onto the same dictionary from the first function.

In [ ]:
def count_cytosines(sequences):
  names=list(sequences.keys())
  for i in range(0,len(sequences)):
    ind=names[i]
    mycode=sequences[ind]["Gene"]
    count_C=mycode.count("C")
    count_A=mycode.count("A")
    count_G=mycode.count("G")
    count_T=mycode.count("T")
    percent_C=(count_C/len(mycode))*100
    percent_A=(count_A/len(mycode))*100
    percent_G=(count_G/len(mycode))*100
    percent_T=(count_T/len(mycode))*100
    # The "+]" motif only seen in the description of PFPs, the word "prion" seen in interacting proteins too so cannot be used
    # for human analysis search for 'amyloid', 'fibril', 'tau'...
    if '+]' in sequences[ind]["Description"]:
      #print(sequences[ind]["Description"])
      sequences[ind]["Prion percent cytosine"]=percent_C
      sequences[ind]["Prion percent adenine"]=percent_A
      sequences[ind]["Prion percent guanine"]=percent_G
      sequences[ind]["Prion percent thymine"]=percent_T
      sequences[ind]["Percent Cytosine"]=0
      sequences[ind]["Percent Adenine"]=0
      sequences[ind]["Percent Guanine"]=0
      sequences[ind]["Percent Thymine"]=0
    else:
      sequences[ind]["Prion percent cytosine"]=0
      sequences[ind]["Prion percent adenine"]=0
      sequences[ind]["Prion percent guanine"]=0
      sequences[ind]["Prion percent thymine"]=0
      sequences[ind]["Percent Cytosine"]=percent_C
      sequences[ind]["Percent Adenine"]=percent_A
      sequences[ind]["Percent Guanine"]=percent_G
      sequences[ind]["Percent Thymine"]=percent_T
  return sequences

def position_bias(sequences):
  names=list(sequences.keys())
  for i in range(0,len(sequences)):
    ind=names[i]
    mycode=list(sequences[ind]["Gene"])
    codons=[sequences[ind]["Gene"][i:i+3] for i in range(0, len(sequences[ind]["Gene"]), 3)]
    first_pos = 0
    second_pos=0
    third_pos=0
    j=0
    leu_coding = codons.count("CTC") + codons.count("CTT")
    leu_noncoding = codons.count("CTA") + codons.count("CTG")
    leu_u=codons.count("TTG") + codons.count("TTA")
    ser_coding=codons.count("TCT") + codons.count("TCC") + codons.count("TCA") + codons.count("TCG")
    ser_noncoding=codons.count("AGT") + codons.count("AGC")
    while j < (len(mycode)-2):
      if mycode[j]=="C":
        first_pos+=1
      j+=1
      if mycode[j]=='C':
        second_pos+=1
      j+=1
      if mycode[j]=='C':
        third_pos+=1
      j+=1

    sequences[ind]["Percent cytosine in first pos"]=(first_pos/(len(mycode)*(1/3)))*100
    sequences[ind]["Percent cytosine in second pos"]=(second_pos/(len(mycode)*(1/3)))*100
    sequences[ind]["Percent cytosine noncoding"]=(third_pos/(len(mycode)*(1/3)))*100
    if leu_noncoding!=0:
      sequences[ind]["Percent leucine vulnerable"]=(leu_coding/(leu_coding+leu_noncoding+leu_u))*100
      sequences[ind]["Percent leucine not vulnerable"]=(leu_noncoding/(leu_coding+leu_noncoding+leu_u))*100
    else:
      sequences[ind]["Percent leucine vulnerable"]=0
      sequences[ind]["Percent leucine  not vulnerable"]=0
    if ser_coding!=0:
      sequences[ind]["Percent serine vulnerable"]=(ser_coding/(ser_coding+ser_noncoding))*100
      sequences[ind]["Percent serine not vulnerable"]=(ser_noncoding/(ser_coding+ser_noncoding))*100
    else:
      sequences[ind]["Percent serine vulnerable"]=0
      sequences[ind]["Percent serine not vulnerable"]=0
  return sequences

In [ ]:
from Bio import SeqIO
import pandas as pd
import numpy as np
import csv

my_dict_raw = SeqIO.to_dict(SeqIO.parse("/content/gdrive/MyDrive/MutantRNAs/yeast/orf_coding_all_R64-4-1_20230830.fasta", "fasta"))
names=list(my_dict_raw.keys())
my_dict={}
for i in range(0,len(my_dict_raw)):
  ind=names[i]
  seq_record=my_dict_raw[ind]
  ID=str(ind)
  Description=str(seq_record.description)
  #print(Description)
  Gene=str(seq_record.seq)
  mydict={"ID":ID,"Description":Description, "Gene":Gene}
  my_dict[ind]=mydict


#print(my_dict.keys())
#print(my_dict.items())
#print(my_dict['YAL030W']['Description'])
#seq=my_dict['YDR172W']["Gene"] #sup35
#prd=seq[0:333].count("C")
#whole=seq.count("C")
#print("cytosines in PRD of sup35: ", (prd/333)*100)
#print("cytosines in complete sup35 protein-coding gene: ", (whole/len(seq))*100)
new=count_cytosines(my_dict)
final=position_bias(new)
#to save all data as a .csv file
#header = ["ID", "Description", "Gene","Percent Cytosine", "Prion percent cytosine","Percent Adenine", "Prion percent adenine",
#"Percent Guanine", "Prion percent guanine", "Percent Thymine", "Prion percent thymine", "Percent cytosine in coding pos", "Percent cytosine noncoding"]
#with open('/content/gdrive/MyDrive/MutantRNAs/human/hdata.csv', 'w') as file:
#    writer = csv.DictWriter(file, fieldnames=header)
#    writer.writeheader()
#    for key, value in final.items():
#        writer.writerow(value)

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mannwhitneyu

prion_percents_cytosine=[]
prion_percents_adenine=[]
prion_percents_guanine=[]
prion_percents_thymine=[]
nonprion_percents_cytosine=[]
nonprion_percents_adenine=[]
nonprion_percents_guanine=[]
nonprion_percents_thymine=[]
total_percent_cytosine=[]
for i in range(0, len(new)):
  ind=names[i]
  total_percent_cytosine.append(new[ind]["Percent Cytosine"])
  if new[ind]["Prion percent cytosine"]==0:
    nonprion_percents_cytosine.append(new[ind]["Percent Cytosine"])
    nonprion_percents_adenine.append(new[ind]["Percent Adenine"])
    nonprion_percents_guanine.append(new[ind]["Percent Guanine"])
    nonprion_percents_thymine.append(new[ind]["Percent Thymine"])
  else:
    prion_percents_cytosine.append(new[ind]["Prion percent cytosine"])
    prion_percents_adenine.append(new[ind]["Prion percent adenine"])
    prion_percents_guanine.append(new[ind]["Prion percent guanine"])
    prion_percents_thymine.append(new[ind]["Prion percent thymine"])

print("Cytosine Prion: ", np.mean(prion_percents_cytosine), "Cytosine nonprion: ", np.mean(nonprion_percents_cytosine))
print("Adenine Prion: ", np.mean(prion_percents_adenine), "Adenine nonprion: ", np.mean(nonprion_percents_adenine))
print("Guanine Prion: ", np.mean(prion_percents_guanine), "Guanine nonprion: ", np.mean(nonprion_percents_guanine))
print("Thymine Prion: ", np.mean(prion_percents_thymine), "Thymine nonprion: ", np.mean(nonprion_percents_thymine))
print(len(prion_percents_cytosine))
fig = plt.figure(figsize =(8,10))
bp1=plt.boxplot([nonprion_percents_adenine, prion_percents_adenine],  showfliers=False, patch_artist=True)
a=plt.gca()
for box in bp1['boxes']:
    # change outline color
    box.set(color='hotpink', linewidth=3)
    # change fill color
    box.set(facecolor = 'blue' )
plt.xticks([1, 2], ['Nonprion', 'Prion'], fontsize=15)
plt.ylabel('Percent Adenine', fontsize=15)
plt.text(1.85,21, 'P value = 0.11', fontsize = 18)
plt.title('Amount of Adenine in Different Sequence Types', fontsize=18)

#plt.savefig("/content/gdrive/MyDrive/MutantRNAs/yeast/adenine.png")
plt.show()
u_statistic_c, p_value_c = mannwhitneyu(prion_percents_cytosine, nonprion_percents_cytosine, alternative='two-sided')
u_statistic_a, p_value_a = mannwhitneyu(prion_percents_adenine, nonprion_percents_adenine, alternative='two-sided')
u_statistic_g, p_value_g = mannwhitneyu(prion_percents_guanine, nonprion_percents_guanine, alternative='two-sided')
u_statistic_t, p_value_t = mannwhitneyu(prion_percents_thymine, nonprion_percents_thymine, alternative='two-sided')
alpha = 0.05
print ("prion ", p_value_c)
print ("nonprion ", p_value_a)
